# Week 1 — Day 2 (20 min): Train/Test Split (and why it matters)

**Goal (20 minutes):** Learn how to split data correctly, avoid common mistakes (like data leakage), and evaluate a model on unseen data.

You’ll:
- Create a **train/test split**
- See what happens when you **don’t split**
- Use **stratify** for classification
- Compare results across different **random states**


In [ ]:
import matplotlib as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

#-----------------------------

# To remove all of the Runtime warnings

import warnings

warnings.filterwarnings("ignore", category = RuntimeWarning)

#-----------------------------

data = load_breast_cancer()
X = data.data
y = data.target

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"Class names : {data.target_names}")
print("Class counts :", {0 : (y == 0).sum(), 1 : (y == 1).sum()})

print("------------------------------------------")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.25,
    random_state = 42,
    stratify = y
)

print(f"Train size : {X_train.shape[0]}")
print(f"Test size : {X_test.shape[0]}")
print("Train class counts :", {0 : (y_train == 0).sum(), 1 : (y_train == 1).sum()})
print("Test class counts :", {0 : (y_test == 0).sum(), 1 : (y_test == 1).sum()})

print("------------------------------------------")

good_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter = 10000))
good_model.fit(X_train, y_train)

y_pred = good_model.predict(X_test)
good_acc = accuracy_score(y_test, y_pred)

print(f"Accuracy on TEST data (realistic) : {good_acc}")

print("------------------------------------------")

def eval_with_seed(seed : int) -> float :
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = 0.25, random_state = seed, stratify = y
    )
    
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
    pipe.fit(X_train, y_train)
    return accuracy_score(y_test, pipe.predict(X_test))

seeds = [0, 1, 2, 42, 99, 102]

for s in seeds:
    print(s, "->", eval_with_seed(s))

X shape : (569, 30)
y shape : (569,)
Class names : ['malignant' 'benign']
Class counts : {0: np.int64(212), 1: np.int64(357)}
------------------------------------------
Train size : 426
Test size : 143
Train class counts : {0: np.int64(159), 1: np.int64(267)}
Test class counts : {0: np.int64(53), 1: np.int64(90)}
------------------------------------------
Accuracy on TEST data (realistic) : 0.986013986013986
------------------------------------------
0 -> 0.958041958041958
1 -> 0.965034965034965
2 -> 0.972027972027972
42 -> 0.986013986013986
99 -> 0.965034965034965
102 -> 0.9790209790209791


## 0) Quick warm-up (1 min)

In one sentence (comment):  
Why do we *not* train and test on the exact same data?


In [ ]:
# Your answer:
# Because if we train and test on the exact same data, it's like making a test and giving the student the same test for him to learn on (basically giving him the answers)


## 1) Load a dataset (3 min)

We’ll use the **Breast Cancer Wisconsin** dataset (binary classification).
- `X`: medical features
- `y`: 0/1 label (benign/malignant)


In [1]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class names:", data.target_names)
print("Class counts:", {0: (y==0).sum(), 1: (y==1).sum()})


X shape: (569, 30)
y shape: (569,)
Class names: ['malignant' 'benign']
Class counts: {0: np.int64(212), 1: np.int64(357)}


## 2) The wrong way (on purpose): Train and test on the same data (4 min)

This often gives **overly optimistic** results because the model “sees” the answers during training.


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

bad_model = LogisticRegression(max_iter=10000)
bad_model.fit(X, y)

bad_pred = bad_model.predict(X)
bad_acc = accuracy_score(y, bad_pred)

print("Accuracy when testing on TRAINING data (misleading):", bad_acc)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linea

Accuracy when testing on TRAINING data (misleading): 0.9578207381370826


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 3) The correct way: Train/Test Split (6 min)

Key ideas:
- Train on one part, test on a **different** part.
- Use `random_state` so results are reproducible.
- Use `stratify=y` so the class ratio stays similar in both splits.


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Train class counts:", {0: (y_train==0).sum(), 1: (y_train==1).sum()})
print("Test class counts:", {0: (y_test==0).sum(), 1: (y_test==1).sum()})


Train size: 426
Test size: 143
Train class counts: {0: np.int64(159), 1: np.int64(267)}
Test class counts: {0: np.int64(53), 1: np.int64(90)}


### Train + evaluate (2 min)


In [8]:
good_model = LogisticRegression(max_iter=10000)
good_model.fit(X_train, y_train)

y_pred = good_model.predict(X_test)
good_acc = accuracy_score(y_test, y_pred)

print("Accuracy on TEST data (realistic):", good_acc)


Accuracy on TEST data (realistic): 0.958041958041958


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linea

## 4) Mini experiment: random_state changes the split (3–6 min)

Run this cell a few times with different `seed` values.  
Notice how accuracy changes slightly because the train/test split changes.


In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def eval_with_seed(seed: int) -> float:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
    pipe.fit(X_train, y_train)
    return accuracy_score(y_test, pipe.predict(X_test))

# Try different seeds here:
seeds = [0, 1, 2, 42, 99, 102, 102]

for s in seeds:
    print(s, "->", eval_with_seed(s))


0 -> 0.958041958041958
1 -> 0.965034965034965
2 -> 0.972027972027972
42 -> 0.986013986013986
99 -> 0.965034965034965
102 -> 0.9790209790209791
102 -> 0.9790209790209791


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_linea